# Two pairs need to agree (CODA-OR)

You have a left pair $(a,b)$ and a right pair $(b',c)$. They become one triple only if $b=b'$ (OR-D1, OR-T1–OR-T3). Putting an object into a comparison envelope is the same kind of move: it organizes pieces. It does not, by itself, grant standing (OR-T6).

This notebook uses the shipped calculator. Arithmetic is exact (fractions, not floating point).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_or_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "or" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "or"
else:
    root = p

assert (root / "code").is_dir(), "run from package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))

from typing import Any, Tuple, Optional
from coda_or_core import (
    ComparisonObject,
    SupportArrow,
    SupportSieve,
    evaluate_comparison_authority,
    R_EMPTY_SUPPORT,
    R_NO_TERMINALIZER,
)
print("CODA-OR core loaded successfully.")

CODA-OR core loaded successfully.


## 1. Overlap Gluing and Incompatible Countermodel (OR-D1, OR-T1–OR-T3)

Two pairs $(a, b)$ and $(b', c)$ glue uniquely to $(a, b, c)$ if and only if $b = b'$.

In [2]:
def glue_pairs(left: Tuple[Any, Any], right: Tuple[Any, Any]) -> Optional[Tuple[Any, Any, Any]]:
    if left[1] == right[0]:
        return (left[0], left[1], right[1])
    return None

# Compatible pairs (OR-T1)
pair_left = ("A_state", "shared_boundary")
pair_right = ("shared_boundary", "C_state")
triple = glue_pairs(pair_left, pair_right)
print("Compatible gluing:", triple)
assert triple == ("A_state", "shared_boundary", "C_state"), "Gluing failed!"

# Incompatible countermodel (OR-T3)
counter_left = ("*", False)
counter_right = (True, "*")
counter_triple = glue_pairs(counter_left, counter_right)
print("Incompatible gluing:", counter_triple)
assert counter_triple is None, "Incompatible pair must refuse gluing!"

Compatible gluing: ('A_state', 'shared_boundary', 'C_state')
Incompatible gluing: None


## 2. Empty Support Refusal (OR-D3, OR-T5)

A comparison object with no incoming certified arrows is refused, even if you fill in descent, verifier, and terminalizer names.

In [3]:
empty = ComparisonObject(
    object_id="X",
    weight_class="finite",
    support_sieve=SupportSieve(target_object_id="X", arrows=()),
    descent_d="d",
    verifier_v="v",
    terminalizer_tau="tau",
)
res = evaluate_comparison_authority(empty)
print("inhabited:", res.inhabited)
print("refusal:", res.refusal_code)
assert res.inhabited is False
assert res.refusal_code == R_EMPTY_SUPPORT

inhabited: False
refusal: R_empty_support


## 3. Support Bound to Target (Review Finding 2)

Support arrows targeting $Y$ cannot be admitted as support for $X$.

In [4]:
import pytest
arrow_y = SupportArrow(
    source_regime_id="A",
    target_object_id="Y",
    source_has_authority=True,
    certificate_id="chi_A",
)
# Attempting to attach arrow targeting Y to sieve targeting X raises ValueError
with pytest.raises(ValueError, match="Arrow target 'Y' does not match sieve target 'X'"):
    SupportSieve(target_object_id="X", arrows=(arrow_y,))
print("Target mismatch between arrow and sieve successfully refused.")

Target mismatch between arrow and sieve successfully refused.


## 4. Declared Certified Support

When the sieve targets $X$, the source has standing, a certificate exists, and $(D, V, \tau)$ are declared, authority is certified (with `AuthorityEffect: None`).

In [5]:
arrow_x = SupportArrow(
    source_regime_id="A",
    target_object_id="X",
    source_has_authority=True,
    certificate_id="chi_A",
)
sieve_x = SupportSieve(target_object_id="X", arrows=(arrow_x,))
valid_obj = ComparisonObject(
    object_id="X",
    weight_class="finite",
    support_sieve=sieve_x,
    descent_d="d",
    verifier_v="v",
    terminalizer_tau="tau",
)
auth_res = evaluate_comparison_authority(valid_obj)
print("inhabited:", auth_res.inhabited)
print("terminal_state:", auth_res.terminal_state)
print("authority_effect:", auth_res.authority_effect)
assert auth_res.inhabited is True
assert auth_res.authority_effect == "None"

inhabited: True
terminal_state: term_certified_comparison_X_tau
authority_effect: None
